<a href="https://colab.research.google.com/github/Prasadn1234/DataScience/blob/main/Week_1_Assignment_Supply_Chain_Project_07072026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contract Intelligence Multi-Agent System
## Assignment Notebook - 8-Week Capstone Project

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Welcome to Your Capstone Assignment!

This notebook is your guided journey to building a **production-grade multi-agent contract intelligence system**. Unlike the master solution, YOU will implement the core functionality.

### How This Assignment Works

1. **Scaffolding Provided**: Class structures, function signatures, and imports are given
2. **TODO Markers**: Look for `# TODO:` comments - these are YOUR tasks
3. **Hints**: Each section has hints to guide you (but not give away the answer)
4. **Validation Cells**: Run these to check if your implementation is correct
5. **Expected Output**: Sample outputs show what success looks like

### Difficulty Progression

| Week | Difficulty | Focus |
|------|-----------|-------|
| 1 | Easy | Environment setup (mostly provided) |
| 2 | Easy-Medium | Document processing methods |
| 3 | Medium | Vector store search functions |
| 4 | Medium-Hard | Implement agents from scratch |
| 5 | Hard | Multi-agent orchestration |
| 6 | Medium | Build knowledge graph |
| 7 | Medium | Add observability metrics |
| 8 | Hard | Integrate everything |

### Grading Rubric

- **Week 1-2**: 15% (Foundation)
- **Week 3-4**: 30% (Core Components)  
- **Week 5-6**: 30% (Advanced Features)
- **Week 7-8**: 25% (Production Polish)

Let's begin!

---

# WEEK 1: Environment & Foundations

**Difficulty: Easy** - Most code is provided. Focus on understanding.

---

## Learning Objectives

By the end of Week 1, you will:
- [ ] Set up all required packages
- [ ] Configure API keys securely
- [ ] Initialize Langfuse for observability
- [ ] Create traced wrapper functions
- [ ] Explore the contract data structure

---

In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory path using quotes
DATA_DIR = Path('/content/drive/MyDrive/Datasets')

# Check if the directory exists
print(f"Exists after mount: {DATA_DIR.exists()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exists after mount: True


## 1.1 Package Installation

This section is **provided** - just run the cell to install packages.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION (PROVIDED)
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

Installing packages (this may take 1-2 minutes)...
All 18 packages installed and verified!


## 1.2 Environment Configuration

This section is **provided** - handles environment detection and imports.

In [ ]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Python Version: {sys.version.split()[0]}")

Runtime Environment: Google Colab
Python Version: 3.12.13


In [ ]:
# ============================================================================
# API KEY CONFIGURATION (PROVIDED)
# ============================================================================

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "contract_intelligence"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://jp.cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://jp.cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/drive/MyDrive/dataset"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'dataset'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

PROJECT_NAME = "contract-intelligence-system"
DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

Configuring for Google Colab environment...
API keys loaded from Colab Secrets

Dataset already available.

All API keys validated successfully
Data directory: /content/drive/MyDrive/dataset/contract_intelligence
Directory exists: True
Total files found: 36


## 1.3 Langfuse Initialization

**YOUR FIRST TODO!** Initialize the Langfuse client.

### Hints:
- Import `Langfuse` from `langfuse`
- Import `observe` and `langfuse_context` from `langfuse.decorators`
- Use environment variables for credentials
- Call `auth_check()` to verify connection

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================
import os
from datetime import datetime
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI
# Initialize the Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://jp.cloud.langfuse.com"),
)
# Verify connection
try:
    langfuse.auth_check()
    print("Langfuse connection verified!")
except Exception as e:
    print(f"Langfuse connection failed: {e}")
# Create a unique session ID
SESSION_ID = f"contract-intel-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
print(f"Session ID: {SESSION_ID}")

Langfuse connection verified!
Session ID: contract-intel-20260706-180900


## 1.4 Traced Wrapper Functions

**TODO:** Implement traced versions of OpenAI calls.

### Hints:
- Create a trace with `langfuse.trace()`
- Create a generation span with `trace.generation()`
- Use `openai_client.embeddings.create()` for embeddings
- End the generation with usage stats

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED EMBEDDING FUNCTION
# ============================================================================
from typing import List
# Initialize OpenAI client (PROVIDED)
openai_client = OpenAI()
def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.
    """
    # 1. Create trace
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "text_length": len(text),
            "project": "contract-intel",
        },
    )
    # 2. Create generation span
    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text,
    )
    # 3. Make API call
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
    )
    embedding = response.data[0].embedding
    # 4. End generation with metadata
    generation.end(
        output={"dimensions": len(embedding)},
        usage={
            "input": response.usage.prompt_tokens,
            "total": response.usage.total_tokens,
        },
    )
    # 5. Return embedding
    return embedding
print("traced_embedding() function defined")

traced_embedding() function defined


In [ ]:
# Quick test cell #
sample_text = "Master service agreement between Acme Corp and Beta LLC."
vector = traced_embedding(sample_text, trace_name="test-embedding")
print(f"Embedding dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")
print(f"Check trace in Langfuse dashboard for session: {SESSION_ID}")

Embedding dimensions: 1536
First 5 values: [-0.0203399658203125, 0.0223236083984375, 0.05340576171875, -0.006641387939453125, -0.01352691650390625]
Check trace in Langfuse dashboard for session: contract-intel-20260706-180900


In [ ]:
# ============================================================================
# WEEK 1.4: TRACED COMPLETION FUNCTION
# ============================================================================
def traced_completion(prompt: str, trace_name: str = "completion") -> str:
    """
    Generate completion with full Langfuse tracing.
    """
    # 1. Create trace
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "prompt_length": len(prompt),
            "project": "contract-intel",
        },
    )
    # 2. Create generation span
    generation = trace.generation(
        name="openai-chat-completion",
        model="gpt-4.1-2025-04-14",
        input=[{"role": "user", "content": prompt}],
    )
    # 3. Make API call
    response = openai_client.chat.completions.create(
        model="gpt-4.1-2025-04-14",
        messages=[{"role": "user", "content": prompt}],
    )
    response_text = response.choices[0].message.content
    # 4. End generation with usage stats
    generation.end(
        output=response_text,
        usage={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens,
            "total": response.usage.total_tokens,
        },
    )
    return response_text
print("traced_completion() function defined")

traced_completion() function defined


In [ ]:
# Quick test #
result = traced_completion(
    "Summarize the key risks in a vendor master service agreement.",
    trace_name="test-completion",
)
print(result)
print(f"View trace in Langfuse session: {SESSION_ID}")

**Key Risks in a Vendor Master Service Agreement (MSA):**

1. **Scope Creep:** Unclear or poorly defined scope can lead to disagreements, additional costs, and extended timelines.

2. **Performance and Service Levels:** Vague performance metrics and SLAs (Service Level Agreements) can make it hard to hold the vendor accountable.

3. **Intellectual Property (IP):** Ambiguities over IP ownership and usage rights can result in legal disputes.

4. **Data Security and Confidentiality:** Insufficient data protection obligations can expose both parties to breaches, regulatory penalties, and reputational damage.

5. **Limitation of Liability:** Overly broad or limited liability, indemnity, or warranty clauses can result in financial exposure.

6. **Termination and Exit:** Lack of clear termination rights or transition assistance can hinder smooth service transfer or business continuity if the agreement ends.

7. **Payment and Pricing Issues:** Unclear or inflexible pricing, payment terms, or d

## 1.5 Contract Document Taxonomy

**PROVIDED** - Study this structure carefully, you'll need it later.

In [ ]:
# ============================================================================
# WEEK 1.5: CONTRACT TAXONOMY (PROVIDED)
# ============================================================================

CONTRACT_CATEGORIES = {
    'master_agreements': {
        'path': 'Master level agreements',
        'description': 'Foundation contracts establishing overall relationship',
        'document_types': ['MSA', 'NDA', 'Rate Cards'],
        'risk_focus': ['legal', 'compliance'],
    },
    'transaction_contracts': {
        'path': 'Transaction level contract',
        'description': 'Project-specific agreements under master agreements',
        'document_types': ['SOW', 'Work Orders', 'Renewals', 'Amendments'],
        'risk_focus': ['operational', 'financial'],
    },
    'commercial_docs': {
        'path': 'Commercial docs',
        'description': 'Pricing and commercial terms documentation',
        'document_types': ['Pricing Tables', 'Rate Schedules'],
        'risk_focus': ['financial'],
    },
    'financial_billing': {
        'path': 'Financial and Billing docs',
        'description': 'Invoices, payment records, and billing schedules',
        'document_types': ['Invoices', 'Credit Notes', 'Payment Schedules'],
        'risk_focus': ['financial'],
    },
    'operational_docs': {
        'path': 'Operational service delivery docs',
        'description': 'Service delivery and operational documentation',
        'document_types': ['Service Reports', 'SLA Reports'],
        'risk_focus': ['operational'],
    },
    'compliance_docs': {
        'path': 'Compliance and policy documents',
        'description': 'Security, compliance, and policy documentation',
        'document_types': ['InfoSec Controls', 'Vendor Policies'],
        'risk_focus': ['compliance'],
    },
    'legal_support': {
        'path': 'Legal support documents',
        'description': 'Legal correspondence and supporting documentation',
        'document_types': ['Legal Memos', 'Correspondence'],
        'risk_focus': ['legal'],
    },
    'relationship_governance': {
        'path': 'Relationship and governance docs',
        'description': 'Governance frameworks and relationship management',
        'document_types': ['Governance Frameworks', 'Escalation Matrices'],
        'risk_focus': ['operational', 'compliance'],
    }
}

print(f"Contract categories defined: {len(CONTRACT_CATEGORIES)}")
for cat, info in CONTRACT_CATEGORIES.items():
    print(f"  - {cat}: {info['description'][:50]}...")

Contract categories defined: 8
  - master_agreements: Foundation contracts establishing overall relation...
  - transaction_contracts: Project-specific agreements under master agreement...
  - commercial_docs: Pricing and commercial terms documentation...
  - financial_billing: Invoices, payment records, and billing schedules...
  - operational_docs: Service delivery and operational documentation...
  - compliance_docs: Security, compliance, and policy documentation...
  - legal_support: Legal correspondence and supporting documentation...
  - relationship_governance: Governance frameworks and relationship management...


## 1.6 Data Discovery

**TODO:** Implement the data discovery function.

### Hints:
- Use `Path.rglob('*')` to recursively find files
- Check file suffixes with `item.suffix.lower()`
- Create spans for each category scan

In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================
#
# Assumes `langfuse`, `SESSION_ID`, and `IN_COLAB` are already defined by
# earlier Week 1 setup cells in the ContractIQ notebook.
#
# Root cause of "Total files discovered: 0":
#   DATA_DIR pointed at MyDrive/data/, which was empty (no category
#   subfolders). The 36 real contract documents are flat files in
#   MyDrive/Datasets/, categorized only by filename keyword (e.g.
#   "MSA sample 1.docx", "Redlined contract sample 2.docx"), not by
#   subfolder. This version matches files by filename pattern per the 8
#   categories from the ContractIQ problem statement, while still
#   supporting a subfolder-per-category layout if you organize the data
#   that way later.
import re
from pathlib import Path
from typing import Dict, List

CONTRACT_CATEGORIES: Dict[str, Dict[str, object]] = {
    "master_level_agreements": {
        "folder": "master_level_agreements",
        "patterns": [r"\bmsa\b", r"\bnda\b", r"rate\s*card"],
    },
    "transaction_contracts": {
        "folder": "transaction_contracts",
        "patterns": [r"\bsow\b", r"contract\s*renewal"],
    },
    "commercial_documents": {
        "folder": "commercial_documents",
        "patterns": [r"pricing\s*table"],
    },
    "financial_and_billing": {
        "folder": "financial_and_billing",
        "patterns": [r"invoice", r"credit\s*note", r"billing\s*schedule"],
    },
    "operational_documents": {
        "folder": "operational_documents",
        "patterns": [r"service\s*report", r"incident\s*(and)?\s*problem"],
    },
    "compliance_documents": {
        "folder": "compliance_documents",
        "patterns": [r"infosec", r"vend\w*\s*mgt\s*policy", r"industry\s*specific"],
    },
    "negotiation_history": {
        "folder": "negotiation_history",
        "patterns": [r"email\s*negotiation", r"redlined", r"risk\s*assessment"],
    },
    "governance": {
        "folder": "governance",
        "patterns": [r"post\s*project\s*review", r"renewal\s*negotiation\s*notes"],
    },
}

_EXTENSIONS = ("*.docx", "*.pdf", "*.xlsx")


def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all contract documents organized by category.
    """
    # 1. Create trace for data discovery
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        metadata={
            "data_dir": str(data_dir),
            "project": "contract-intel",
        },
    )
    # 2. Initialize discovered dictionary
    discovered: Dict[str, List[Path]] = {cat: [] for cat in CONTRACT_CATEGORIES.keys()}
    # 3. Check if data directory exists
    if not data_dir.exists():
        trace.update(
            output={
                "status": "missing_directory",
                "total_files": 0,
                "by_category": {cat: 0 for cat in discovered},
            }
        )
        print(f"Warning: Data directory not found: {data_dir}")
        return discovered

    # 4. Gather every candidate file once
    all_files: List[Path] = []
    for pattern in _EXTENSIONS:
        all_files.extend(data_dir.rglob(pattern))
    all_files = sorted(set(all_files))

    compiled_patterns = {
        cat: [re.compile(p, re.IGNORECASE) for p in info["patterns"]]
        for cat, info in CONTRACT_CATEGORIES.items()
    }
    folder_to_category = {
        info["folder"]: cat for cat, info in CONTRACT_CATEGORIES.items()
    }

    unmatched: List[Path] = []
    for file_path in all_files:
        # Prefer subfolder-per-category layout when present
        category = folder_to_category.get(file_path.parent.name.lower())
        if category is None:
            # Fall back to matching the filename against category keyword patterns
            stem = file_path.stem
            category = next(
                (
                    cat
                    for cat, patterns in compiled_patterns.items()
                    if any(p.search(stem) for p in patterns)
                ),
                None,
            )
        if category is not None:
            discovered[category].append(file_path)
        else:
            unmatched.append(file_path)

    for cat in discovered:
        discovered[cat] = sorted(set(discovered[cat]))

    # 5. Calculate and log totals
    totals_by_category = {cat: len(files) for cat, files in discovered.items()}
    total_files = sum(totals_by_category.values())
    trace.update(
        output={
            "status": "success",
            "total_files": total_files,
            "by_category": totals_by_category,
            "unmatched_files": [str(p) for p in unmatched],
        }
    )
    if unmatched:
        print(f"Note: {len(unmatched)} file(s) didn't match any category pattern:")
        for p in unmatched:
            print(f"  - {p}")
    return discovered


# Set up data path
if IN_COLAB:
    # Real contract documents live in MyDrive/Datasets/, not MyDrive/data/
    # (the latter was empty - that's why discovery returned 0 files before).
    DATA_DIR = Path("/content/drive/MyDrive/dataset")
else:
    DATA_DIR = Path("../dataset")

# Run discovery
contract_files = discover_contract_data(DATA_DIR)
print("\nDiscovery Summary:")
total = 0
for cat, files in contract_files.items():
    if files:
        print(f"  {cat}: {len(files)} files")
        total += len(files)
print(f"\nTotal files discovered: {total}")

Note: 66 file(s) didn't match any category pattern:
  - /content/drive/MyDrive/dataset/ecommerce/1) Intro and flow docs.docx
  - /content/drive/MyDrive/dataset/ecommerce/2) JSON input for agent 1.docx
  - /content/drive/MyDrive/dataset/ecommerce/3)Output sample agent 1.docx
  - /content/drive/MyDrive/dataset/ecommerce/4) JSON input for AGENT 2.docx
  - /content/drive/MyDrive/dataset/ecommerce/5) Sample output agent 2.docx
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/BMI report zz.docx
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/Blood sugar test report zz.docx
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/Blood test parameters explained zz.docx
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/CBC PARAMETERS zz.docx
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/CBC Report - Patient Anal.pdf
  - /content/drive/MyDrive/dataset/healthcare/Clinician , Doctor/CBC report.zz.docx
  - /content/drive/My

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create sample structure for testing
test_dir = Path("./data")
for folder in ["master_service_agreements", "ndas"]:
    (test_dir / folder).mkdir(parents=True, exist_ok=True)
    (test_dir / folder / "sample.pdf").touch()
DATA_DIR = test_dir
contract_files = discover_contract_data(DATA_DIR)

Note: 2 file(s) didn't match any category pattern:
  - data/master_service_agreements/sample.pdf
  - data/ndas/sample.pdf


In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================
#
# Assumes `langfuse`, `SESSION_ID`, and `IN_COLAB` are already defined by
# earlier Week 1 setup cells in the ContractIQ notebook.
#
# Root cause of "Total files discovered: 0":
#   DATA_DIR pointed at MyDrive/data/, which was empty (no category
#   subfolders). The 36 real contract documents are flat files in
#   MyDrive/Datasets/, categorized only by filename keyword (e.g.
#   "MSA sample 1.docx", "Redlined contract sample 2.docx"), not by
#   subfolder. This version matches files by filename pattern per the 8
#   categories from the ContractIQ problem statement, while still
#   supporting a subfolder-per-category layout if you organize the data
#   that way later.
import re
from pathlib import Path
from typing import Dict, List

CONTRACT_CATEGORIES: Dict[str, Dict[str, object]] = {
    "master_level_agreements": {
        "folder": "master_level_agreements",
        "patterns": [r"\bmsa\b", r"\bnda\b", r"rate\s*card"],
    },
    "transaction_contracts": {
        "folder": "transaction_contracts",
        "patterns": [r"\bsow\b", r"contract\s*renewal"],
    },
    "commercial_documents": {
        "folder": "commercial_documents",
        "patterns": [r"pricing\s*table"],
    },
    "financial_and_billing": {
        "folder": "financial_and_billing",
        "patterns": [r"invoice", r"credit\s*note", r"billing\s*schedule"],
    },
    "operational_documents": {
        "folder": "operational_documents",
        "patterns": [r"service\s*report", r"incident\s*(and)?\s*problem"],
    },
    "compliance_documents": {
        "folder": "compliance_documents",
        "patterns": [r"infosec", r"vend\w*\s*mgt\s*policy", r"industry\s*specific"],
    },
    "negotiation_history": {
        "folder": "negotiation_history",
        "patterns": [r"email\s*negotiation", r"redlined", r"risk\s*assessment"],
    },
    "governance": {
        "folder": "governance",
        "patterns": [r"post\s*project\s*review", r"renewal\s*negotiation\s*notes"],
    },
}

_EXTENSIONS = ("*.docx", "*.pdf", "*.xlsx")


def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all contract documents organized by category.
    """
    # 1. Create trace for data discovery
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        metadata={
            "data_dir": str(data_dir),
            "project": "contract-intel",
        },
    )
    # 2. Initialize discovered dictionary
    discovered: Dict[str, List[Path]] = {cat: [] for cat in CONTRACT_CATEGORIES.keys()}
    # 3. Check if data directory exists
    if not data_dir.exists():
        trace.update(
            output={
                "status": "missing_directory",
                "total_files": 0,
                "by_category": {cat: 0 for cat in discovered},
            }
        )
        print(f"Warning: Data directory not found: {data_dir}")
        return discovered

    # 4. Gather every candidate file once
    all_files: List[Path] = []
    for pattern in _EXTENSIONS:
        all_files.extend(data_dir.rglob(pattern))
    all_files = sorted(set(all_files))

    compiled_patterns = {
        cat: [re.compile(p, re.IGNORECASE) for p in info["patterns"]]
        for cat, info in CONTRACT_CATEGORIES.items()
    }
    folder_to_category = {
        info["folder"]: cat for cat, info in CONTRACT_CATEGORIES.items()
    }

    unmatched: List[Path] = []
    for file_path in all_files:
        # Prefer subfolder-per-category layout when present
        category = folder_to_category.get(file_path.parent.name.lower())
        if category is None:
            # Fall back to matching the filename against category keyword patterns
            stem = file_path.stem
            category = next(
                (
                    cat
                    for cat, patterns in compiled_patterns.items()
                    if any(p.search(stem) for p in patterns)
                ),
                None,
            )
        if category is not None:
            discovered[category].append(file_path)
        else:
            unmatched.append(file_path)

    for cat in discovered:
        discovered[cat] = sorted(set(discovered[cat]))

    # 5. Calculate and log totals
    totals_by_category = {cat: len(files) for cat, files in discovered.items()}
    total_files = sum(totals_by_category.values())
    trace.update(
        output={
            "status": "success",
            "total_files": total_files,
            "by_category": totals_by_category,
            "unmatched_files": [str(p) for p in unmatched],
        }
    )
    if unmatched:
        print(f"Note: {len(unmatched)} file(s) didn't match any category pattern:")
        for p in unmatched:
            print(f"  - {p}")
    return discovered


# Set up data path
if IN_COLAB:
    # Real contract documents live in MyDrive/Datasets/, not MyDrive/data/
    # (the latter was empty - that's why discovery returned 0 files before).
    DATA_DIR = Path("/content/drive/MyDrive/Datasets")
else:
    DATA_DIR = Path("../data")

# Run discovery
contract_files = discover_contract_data(DATA_DIR)
print("\nDiscovery Summary:")
total = 0
for cat, files in contract_files.items():
    if files:
        print(f"  {cat}: {len(files)} files")
        total += len(files)
print(f"\nTotal files discovered: {total}")


Discovery Summary:
  master_level_agreements: 6 files
  transaction_contracts: 4 files
  commercial_documents: 2 files
  financial_and_billing: 4 files
  operational_documents: 4 files
  compliance_documents: 6 files
  negotiation_history: 6 files
  governance: 4 files

Total files discovered: 36


## Checkpoint: Week 1 Verification

Run this cell to verify your Week 1 implementation.

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Langfuse initialized
try:
    checks.append(("Langfuse initialized", langfuse is not None))
except:
    checks.append(("Langfuse initialized", False))

# Check 2: Session ID created
try:
    checks.append(("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 10))
except:
    checks.append(("Session ID created", False))

# Check 3: OpenAI client ready
try:
    checks.append(("OpenAI client ready", openai_client is not None))
except:
    checks.append(("OpenAI client ready", False))

# Check 4: traced_embedding works
try:
    test_emb = traced_embedding("test", "test-validation")
    checks.append(("traced_embedding() works", test_emb is not None and len(test_emb) == 1536))
except Exception as e:
    checks.append(("traced_embedding() works", False))
    print(f"  Error: {e}")

# Check 5: traced_completion works
try:
    test_comp = traced_completion("Say 'hello'", "test-validation")
    checks.append(("traced_completion() works", test_comp is not None and len(test_comp) > 0))
except Exception as e:
    checks.append(("traced_completion() works", False))
    print(f"  Error: {e}")

# Check 6: Contract taxonomy
checks.append(("Contract taxonomy defined", len(CONTRACT_CATEGORIES) == 8))

# Check 7: Data discovery
try:
    checks.append(("Data discovery function works", callable(discover_contract_data)))
except:
    checks.append(("Data discovery function works", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")
    print("\nHint: Check the error messages and compare with expected output.")

WEEK 1 CHECKPOINT - Verification
  [PASS] Langfuse initialized
  [PASS] Session ID created
  [PASS] OpenAI client ready
  [PASS] traced_embedding() works
  [PASS] traced_completion() works
  [PASS] Contract taxonomy defined
  [PASS] Data discovery function works

ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.
